# 01 — Exploratory Data Analysis

Run against the real extract (`data/raw/articles.csv`, 97,147 rows as of the last
extraction). Covers: null cleanup, class balance by site, title length
distribution, tags population, and a blocked check (date/month coverage —
see cell below) pending a re-export that adds a date column.

**Run this in Colab or locally** — no GPU needed, this is pandas-only.
If running in Colab, mount Drive first and adjust the path in the next cell.

In [ ]:
import sys
sys.path.append("..")   # so `from src...` imports work when running from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.data.extract import validate_extract

# If running in Colab with Drive mounted, change this to your Drive path, e.g.:
# CSV_PATH = "/content/drive/MyDrive/dl_final_project/data/raw/articles.csv"
CSV_PATH = "../data/raw/articles.csv"

df = validate_extract(CSV_PATH)

## 1. Drop rows with missing `teaser_title`

Confirmed small (274 rows, 0.28%) — decision was to drop rather than investigate
further or fall back to `item_name`.

In [ ]:
n_before = len(df)
df = df.dropna(subset=["teaser_title"]).reset_index(drop=True)
n_after = len(df)
print(f"Dropped {n_before - n_after} rows with null teaser_title "
      f"({(n_before - n_after) / n_before:.2%}). Remaining: {n_after:,}")

## 2. Class balance by site

Target is constructed to be ~10% positive *within* each site+month bucket —
this checks that holds in practice, and shows the raw article-count split
between mako and n12 (already known to be imbalanced: 61,653 vs 35,494 at
extraction time, before the teaser_title drop above).

In [ ]:
site_summary = df.groupby("site")["target"].agg(count="count", positive_rate="mean")
print(site_summary)

site_summary["count"].plot(kind="bar", title="Article count by site")
plt.ylabel("count")
plt.show()

## 3. Title length distribution

Both character count and whitespace-token count — the token count is the more
useful number for choosing a tokenizer `max_length` later (Hebrew word
boundaries won't exactly match SigLIP2's subword tokenizer, but this gives a
reasonable estimate to sanity-check against).

In [ ]:
title_char_len = df["teaser_title"].str.len()
title_word_len = df["teaser_title"].str.split().str.len()

print("Character length:")
print(title_char_len.describe())
print("\nWord count:")
print(title_word_len.describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
title_char_len.hist(bins=40, ax=axes[0])
axes[0].set_title("Title length (characters)")
title_word_len.hist(bins=30, ax=axes[1])
axes[1].set_title("Title length (words)")
plt.tight_layout()
plt.show()

## 4. Tags population

Not used in the core model (title+image+site) — kept here for the later
enrichment check (`core_model_plus_tags` in configs/base_config.yaml).

**First inspect the raw format** before parsing — don't assume a separator
(comma vs. pipe vs. JSON array) without looking at real examples first.

In [ ]:
n_null_tags = df["tags"].isna().sum()
print(f"Rows with NO tags at all: {n_null_tags:,} ({n_null_tags / len(df):.1%})")
print(f"Rows WITH tags: {df['tags'].notna().sum():,} ({df['tags'].notna().mean():.1%})")

print("\nRaw examples (inspect the format before assuming a separator):")
for val in df["tags"].dropna().head(10):
    print(repr(val))

In [ ]:
# Best-effort tag-count parser -- ADJUST THE SEPARATOR based on what you saw above.
# This assumes comma-separated; change to "|" or json.loads(...) if the real
# format printed above looks different.
SEPARATOR = ","

tag_counts = df["tags"].dropna().apply(lambda s: len([t for t in s.split(SEPARATOR) if t.strip()]))
print(tag_counts.describe())
tag_counts.hist(bins=20)
plt.title("Tags per article (non-null rows)")
plt.show()

## 5. Date / month coverage

`display_time` is now in the export. Derive `publish_month` here (not
in SQL) and check for gaps before committing to train/val/test month
boundaries -- a month with very few rows would make a bad split point.

In [ ]:
df["display_time"] = pd.to_datetime(df["display_time"])
df["publish_month"] = df["display_time"].dt.to_period("M").astype(str)

monthly_counts = df.groupby(["publish_month", "site"]).size().unstack(fill_value=0)
print(monthly_counts)

monthly_counts.plot(kind="bar", stacked=True, figsize=(10, 4))
plt.title("Articles per month, by site")
plt.ylabel("count")
plt.tight_layout()
plt.show()

# Flag any month that looks thin relative to the others -- a likely
# candidate for a partial/incomplete month at either edge of the date range.
monthly_totals = monthly_counts.sum(axis=1)
thin_months = monthly_totals[monthly_totals < monthly_totals.median() * 0.5]
if len(thin_months) > 0:
    print("\n\u26a0\ufe0f  Months with unusually low counts (check if partial/incomplete):")
    print(thin_months)

## 6. Publish hour vs. target

Does time-of-day at publication correlate with success? Note: `target` is
already normalized within site+month, so this isn't about raw traffic
patterns by hour (e.g. more people online at 9pm) -- it's specifically
about whether publishing at a given hour predicts *relative* outperformance
vs. other articles on the same site that month. Also printing article
counts per hour alongside the rate -- an hour with very few articles can
produce a noisy/unreliable rate that shouldn't be over-interpreted.

In [ ]:
df["publish_hour"] = df["display_time"].dt.hour

hourly = df.groupby("publish_hour")["target"].agg(count="count", positive_rate="mean")
print(hourly)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(hourly.index, hourly["positive_rate"], color="steelblue", alpha=0.7)
ax1.set_xlabel("Publish hour (0-23)")
ax1.set_ylabel("Positive rate", color="steelblue")
ax1.set_title("Positive rate by publish hour (bars) vs. article count (line)")

ax2 = ax1.twinx()
ax2.plot(hourly.index, hourly["count"], color="darkorange", marker="o")
ax2.set_ylabel("Article count", color="darkorange")
plt.show()

print("\nHours with the fewest articles (rates here are the least reliable):")
print(hourly.sort_values("count").head(5))

## 7. Daily volume — central tendency and outlier days

Mean/median/std of articles published per day, and flagging days that sit
far from the average (using the IQR method rather than a fixed cutoff --
more robust to the skew a raw mean/std would have if a few days are extreme).

In [ ]:
df["publish_date"] = df["display_time"].dt.date
daily_counts = df.groupby("publish_date").size()

print(daily_counts.describe())

q1, q3 = daily_counts.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outlier_days = daily_counts[(daily_counts < lower_bound) | (daily_counts > upper_bound)]

print(f"\nIQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"Outlier days (n={len(outlier_days)}):")
print(outlier_days.sort_values())

daily_counts.plot(figsize=(12, 4), title="Articles per day")
plt.axhline(daily_counts.median(), color="green", linestyle="--", label="median")
plt.axhline(lower_bound, color="red", linestyle=":", label="IQR bounds")
plt.axhline(upper_bound, color="red", linestyle=":")
plt.legend()
plt.show()